<a href="https://colab.research.google.com/github/LivingstonTardzenyuy/Generative-AI/blob/main/working_with_custom_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import os
from google.colab import userdata

grok_key = userdata.get('GROK_api')

In [13]:
!pip install langchain-groq
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model = "openai/gpt-oss-20b",
    groq_api_key = grok_key
)

In [16]:
messages = [
    ("system", "You are a historian expert in the Kennedy family."),
    ("user", "who is Kongnyuy Livingston ?"),
]

response = chatModel.invoke(messages)
response.content

'I’m not aware of any historical figure named **“Kongnyuy Livingston.”** It’s possible that the name is misspelled, a transcription error, or perhaps a very obscure individual whose records aren’t part of the mainstream historical record.\n\nIf you have any additional context—such as a time period, a location, a related event, or how you came across the name—I’d be glad to dig deeper. In the meantime, here are a few points that might help clarify the situation:\n\n| Possibility | What it could be | Why it matters |\n|-------------|------------------|----------------|\n| **Misspelling of “Kongnyu”** | “Kongnyu” is a Korean given name (often romanized as “Kong-nyu” or “Gong-nyu”). | If the person is Korean, the surname “Livingston” would be unusual unless it’s an adopted or anglicized name. |\n| **Typo for “Kongnuy” or “Kongny”** | Perhaps the name was mis‑typed in a source. | A small typo can lead to a completely different search result. |\n| **Fictional or Contemporary** | The name mig

## From the above we see the limitations with our LLM.
it does not have access to many much information.

Now we will have to train it with our own custom data.

# Getting some data from drive and loading it.

In [19]:
# Install gdown library if not already installed
!pip install gdown

In [20]:
import os
import gdown

# The folder 'data' should already exist from previous steps
folder_name = 'data'

# List of files to download
files_to_download = [
    {
        'url': 'https://docs.google.com/document/d/1GgsuYbVOMhFX485g9DfQCCga5F0wFupC2xye-C7J5QQ/edit?tab=t.0',
        'type': 'doc',
        'output_name': 'document1.txt'
    },
    {
        'url': 'https://docs.google.com/spreadsheets/d/1a-UcUgnd2YFqb8pj0bKMxN9FEk9qNO4qJKYTnoWjsdA/edit?gid=0#gid=0',
        'type': 'sheet',
        'output_name': 'spreadsheet1.csv'
    },
    {
        'url': 'https://docs.google.com/document/d/1EeLapuw8QVuVFSAkeCETGKOPi0yUT7PaJ0wgaF8hD7A/edit?tab=t.0#heading=h.mbra3ggubsrw',
        'type': 'doc',
        'output_name': 'document2.txt'
    }
]

for file_info in files_to_download:
    url = file_info['url']
    file_type = file_info['type']
    output_name = file_info['output_name']

    # Extract document/spreadsheet ID
    if 'document/d/' in url:
        document_id = url.split('document/d/')[1].split('/')[0]
    elif 'spreadsheets/d/' in url:
        document_id = url.split('spreadsheets/d/')[1].split('/')[0]
    else:
        print(f"Skipping {url}: Unrecognized Google Drive URL format.")
        continue

    # Construct export URL based on file type
    if file_type == 'doc':
        export_url = f'https://docs.google.com/document/d/{document_id}/export?format=txt'
    elif file_type == 'sheet':
        export_url = f'https://docs.google.com/spreadsheets/d/{document_id}/export?format=csv'
    else:
        print(f"Skipping {url}: Unsupported file type {file_type}.")
        continue

    # Define the full path for the downloaded file
    output_file_path = os.path.join(folder_name, output_name)

    print(f"\nDownloading {output_name} from {url}...")
    gdown.download(export_url, output_file_path, quiet=False)
    print(f"File downloaded to: {output_file_path}")

/usr/local/lib/python3.12/dist-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=None
  warnings.warn(
Downloading...
From: https://docs.google.com/document/d/1GgsuYbVOMhFX485g9DfQCCga5F0wFupC2xye-C7J5QQ/export?format=txt
To: /content/data/document1.txt
7.48kB [00:00, 4.08MB/s]


File downloaded to: data/document1.txt



Downloading...
From: https://docs.google.com/spreadsheets/d/1a-UcUgnd2YFqb8pj0bKMxN9FEk9qNO4qJKYTnoWjsdA/export?format=csv
To: /content/data/spreadsheet1.csv
8.96kB [00:00, 1.06MB/s]


File downloaded to: data/spreadsheet1.csv



Downloading...
From: https://docs.google.com/document/d/1EeLapuw8QVuVFSAkeCETGKOPi0yUT7PaJ0wgaF8hD7A/export?format=txt
To: /content/data/document2.txt
8.95kB [00:00, 6.31MB/s]

File downloaded to: data/document2.txt


You can verify the downloaded files by listing the contents of the `data` folder:

In [22]:
import os

folder_name = 'data'
print(f"Contents of '{folder_name}' folder:")
for filename in os.listdir(folder_name):
    print(filename)

Contents of 'data' folder:
document2.txt
document1.txt
spreadsheet1.csv


# Data Loader

In [23]:
!pip install langchain_community==0.2.10

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: langsmith
    Found existing install

In [24]:
# txt data loading.

from langchain_community.document_loaders import TextLoader

loader1 = TextLoader("data/document1.txt")
loader2 = TextLoader("data/document2.txt")


# Load the data.
documents1 = loader1.load()
documents2 = loader2.load()

In [27]:
documents1

[Document(metadata={'source': 'data/document1.txt'}, page_content='\ufeffCAITCC Vocational Training Program\n\n\nModule: Introduction to Prompt Engineering\n\n\nPrepared by: Kongnyuy Livingston & Nyuydini Bill\n\n\nFor: CAITCC Vocational Training Center\n\n\nPrograms: AI Product Manager | AI Product Marketing Manager | AI Engineer\nPhase 1: Understanding the Basics of Programming (Using Python)\n(For CAITCC Vocational Training – AI Product Management Program)\n________________\n\n\n1. Introduction to Programming\nProgramming means giving a computer a set of instructions to perform a specific task.\nIt’s like teaching a computer to solve a problem step by step.\nJust as we use English or French to talk to people, programmers use languages like Python to communicate with computers.\n💡 Why Python?\nPython is a beginner-friendly programming language that is:\n* Simple and readable — it looks almost like English.\n\n* Used everywhere — in AI, data science, web development, and automation.\n

In [29]:
# Loading csv files.
from langchain_community.document_loaders import CSVLoader

loader3 = CSVLoader("data/spreadsheet1.csv")
loaded_data = loader3.load()
loaded_data

[Document(metadata={'source': 'data/spreadsheet1.csv', 'row': 0}, page_content='<!DOCTYPE html><style nonce="NeeZTuIZR47O7qPnS5qZCg">body{height:100%;margin:0;width:100%}@media (max-height:350px){.button{font-size:10px}.button-container{margin-top:16px}.button.primary-button: /*# sourceMappingURL=style.css.map */</style><script nonce="NNciLy2_ZyMwemaiwcR75A">\'use strict\';function h(a){var b=0;return function(){return b<a.length?{done:!1\n.button.primary-button:active: None\n.button.primary-button:focus: None\n.button.primary-button:hover{padding:4px 12px}.title-text{font-size:22px;line-height:24px}.subtitle-text{font-size:12px;line-height:18px}}@media (min-height:350px){.button{font-size:14px}.button-container{margin-top:16px}.button.primary-button: login_counter];function m(a\n.button.primary-button:hover{padding:12px 24px}.title-text{font-size:28px;line-height:36px}.subtitle-text{font-size:16px;line-height:24px}}.document-root{display:-webkit-box;display:-webkit-flex;display:-moz-b